# Awards

## Objective

Create team-level awards features for each World Cup. The final output should be one row per team per tournament, with points summed separately for each award/trophy type. FullRoster is used as the base table so only players on that World Cup roster can contribute points.

## Inputs

- `1.DataCleaning-R/Data/CSV/Awards.csv`
- `1.DataCleaning-R/Data/RDS/FullRoster.rds`

## Output

- `1.DataCleaning-R/Data/RDS/Awards.rds`

## Packages


In [10]:
library(tidyverse)
library(here)

## Load Data

Read award points and the full World Cup roster. The roster is the source of truth for which player can count for a team in a given tournament.


In [11]:
awards <- read.csv(here("1.DataCleaning-R", "Data", "CSV", "Awards.csv"))

FullRoster <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullRoster.rds"))


## Aggregate To Team-Tournament

Join award points onto rostered players, replace missing award values with zero, then sum each trophy/award point column by team and World Cup.


In [12]:
award_point_cols <- c("ballon_dor_points", "kopa_points", "yashin_points", "uefa_defender_points", "total_points")

AwardsByRoster <- FullRoster %>%
   left_join(awards, by = c("full_name" = "player", "tournament_id" = "world_cup")) %>%
   mutate(across(all_of(award_point_cols), ~ replace_na(.x, 0)))

awards <- AwardsByRoster %>%
   group_by(tournament_id, team_name, team_id, team_code) %>%
   summarize(across(all_of(award_point_cols), sum), .groups = "drop") %>%
   arrange(tournament_id, desc(total_points))


## Inspect

Quickly inspect the resulting awards table before saving.


In [13]:

awards

tournament_id,team_name,team_id,team_code,ballon_dor_points,kopa_points,yashin_points,uefa_defender_points,total_points
<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<int>,<dbl>
WC-2010,Argentina,T-03,ARG,1009,0,0,0,1009
WC-2010,Portugal,T-58,PRT,956,0,0,0,956
WC-2010,Spain,T-73,ESP,394,0,0,0,394
WC-2010,Ivory Coast,T-42,CIV,144,0,0,0,144
WC-2010,England,T-28,ENG,135,0,0,0,135
WC-2010,Cameroon,T-11,CMR,81,0,0,0,81
WC-2010,Italy,T-41,ITA,58,0,0,0,58
WC-2010,France,T-30,FRA,15,0,0,0,15
WC-2010,Brazil,T-09,BRA,14,0,0,0,14


## Save

Persist the cleaned awards table for downstream modeling notebooks.


In [14]:
saveRDS(awards, here("1.DataCleaning-R", "Data", "RDS", "Awards.rds"))